# IFRS 9 Expected Credit Loss (ECL)

## Objective

The objective of this notebook is to estimate the Expected Credit Loss (ECL) for the retail lending portfolio under the IFRS 9 impairment framework.

The ECL calculation combines the Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD), while incorporating forward-looking macroeconomic scenarios developed in the previous notebook.

This implementation adopts a simplified IFRS 9 methodology suitable for demonstration purposes:

- **Stage 1:** Performing loans using 12-month PD.
- **Stage 2:** Loans with Significant Increase in Credit Risk (SICR) using Lifetime PD.
- **Stage 3:** Credit-impaired loans with PD assumed to be 100%.

The assumptions adopted in this project are:

- Constant LGD = **45%**
- EAD = Outstanding Loan Balance 
- Lifetime PD is approximated using the cumulative default probability over the remaining loan term.
- Portfolio ECL is calculated under Baseline, Downside, and Severe macroeconomic scenarios and combined using scenario probability weights.

This notebook completes the end-to-end IFRS 9 Expected Credit Loss framework by translating macroeconomic stress scenarios into portfolio credit loss estimates.

In [46]:
import pandas as pd
import numpy as np

In [47]:
DATA = "../outputs/loan_level_pd.csv"
df = pd.read_csv(DATA)
df_ecl = df.copy()

In [48]:
df_ecl["IFRS9_Stage"] = "Stage 1"

stage2_status = [
    "Late (16-30 days)",
    "Late (31-120 days)",
    "In Grace Period"
]

stage3_status = [
    "Charged Off"
]

df_ecl.loc[
    df_ecl["loan_status"].isin(stage2_status),
    "IFRS9_Stage"
] = "Stage 2"

df_ecl.loc[
    df_ecl["loan_status"].isin(stage3_status),
    "IFRS9_Stage"
] = "Stage 3"

In [49]:
stage_summary = (
    df_ecl["IFRS9_Stage"]
    .value_counts()
    .rename_axis("Stage")
    .reset_index(name="Loans")
)

stage_summary["Percent"] = (
    stage_summary["Loans"]
    / stage_summary["Loans"].sum()
    *100
)

stage_summary

,Stage,Loans,Percent
0,Stage 1,9822,98.22
1,Stage 2,171,1.71
2,Stage 3,7,0.07


In [50]:
LGD = 0.45

df_ecl["LGD"] = LGD

In [51]:
df_ecl["EAD"] = (
    df_ecl["loan_amount"]
    - df_ecl["paid_principal"]
).clip(lower=0)

In [52]:
df_ecl["Remaining_Years"] = (
    df_ecl["term"] / 12
)


In [53]:
def lifetime_pd(pd_12m, years):
    """
    Convert 12-month PD into Lifetime PD
    assuming a constant annual hazard rate.
    """
    return 1 - (1 - pd_12m) ** years

In [54]:
scenario_pd = {
    "Baseline": "PD_Baseline",
    "Downside": "PD_Downside",
    "Severe": "PD_Severe"
}

for scenario, pd_column in scenario_pd.items():

    # Stage 1 = 12-month PD
    stage1_pd = df_ecl[pd_column]

    # Stage 2 = Lifetime PD
    stage2_pd = lifetime_pd(
        stage1_pd,
        df_ecl["Remaining_Years"]
    )

    # Stage 3 = Defaulted
    stage3_pd = 1.0

    df_ecl[f"PD_{scenario}"] = np.select(
        [
            df_ecl["IFRS9_Stage"] == "Stage 1",
            df_ecl["IFRS9_Stage"] == "Stage 2",
            df_ecl["IFRS9_Stage"] == "Stage 3"
        ],
        [
            stage1_pd,
            stage2_pd,
            stage3_pd
        ]
    )
    
    df_ecl[f"PD_{scenario}"] = (
        df_ecl[f"PD_{scenario}"]
        .clip(lower=0, upper=1)
    )


    df_ecl[f"ECL_{scenario}"] = (
        df_ecl[f"PD_{scenario}"]
        * df_ecl["LGD"]
        * df_ecl["EAD"]
    )

In [55]:
portfolio_ecl = pd.DataFrame({
    "Scenario": scenario_pd.keys(),
    "Portfolio_ECL": [
        df_ecl[f"ECL_{s}"].sum()
        for s in scenario_pd.keys()
    ]
})

print(portfolio_ecl)

   Scenario  Portfolio_ECL
0  Baseline   6.371756e+05
1  Downside   1.358213e+06
2    Severe   1.589669e+06


In [56]:
scenario_weight = {
    "Baseline":0.60,
    "Downside":0.30,
    "Severe":0.10
}

df_ecl["Weighted_ECL"] = 0

for s in scenario_weight.keys():

    df_ecl["Weighted_ECL"] += (
        df_ecl[f"ECL_{s}"]
        * scenario_weight[s]
    )


In [57]:
portfolio_ifrs9 = df_ecl["Weighted_ECL"].sum()

print(f"Portfolio IFRS 9 ECL = ${portfolio_ifrs9:,.2f}")

Portfolio IFRS 9 ECL = $948,736.22


In [58]:
stage_ecl = (
    df_ecl.groupby("IFRS9_Stage")["Weighted_ECL"]
    .agg(["count","sum","mean"])
    .rename(columns={
        "count":"Loans",
        "sum":"Total_ECL",
        "mean":"Average_ECL"
    })
)

print(stage_ecl)

             Loans      Total_ECL  Average_ECL
IFRS9_Stage                                   
Stage 1       9822  755998.329361    76.969897
Stage 2        171  154229.481001   901.926789
Stage 3          7   38508.408000  5501.201143


In [59]:
portfolio_summary = pd.DataFrame({
    "Metric":[
        "Number of Loans",
        "Total Portfolio ECL",
        "Average ECL per Loan"
    ],
    "Value":[
        len(df_ecl),
        df_ecl["Weighted_ECL"].sum(),
        df_ecl["Weighted_ECL"].mean()
    ]
})

print(portfolio_summary)

                 Metric          Value
0       Number of Loans   10000.000000
1   Total Portfolio ECL  948736.218361
2  Average ECL per Loan      94.873622


In [60]:
portfolio_total = df_ecl["Weighted_ECL"].sum()
stage_total = stage_ecl["Total_ECL"].sum()

print(f"Portfolio ECL : {portfolio_total:,.2f}")
print(f"Stage ECL Total: {stage_total:,.2f}")
print(f"Difference     : {portfolio_total-stage_total:.6f}")

Portfolio ECL : 948,736.22
Stage ECL Total: 948,736.22
Difference     : 0.000000


## Business Interpretation

The Expected Credit Loss (ECL) estimates the amount of credit losses expected under different macroeconomic conditions. As economic conditions deteriorate from the Baseline to Severe scenario, higher probabilities of default lead to increased portfolio credit losses.

The three-stage IFRS 9 impairment framework reflects differing levels of credit risk:

- **Stage 1** loans recognise 12-month expected losses for performing exposures.
- **Stage 2** loans recognise lifetime expected losses following a significant increase in credit risk.
- **Stage 3** loans are considered credit-impaired, with expected losses based primarily on the recoverable portion of the exposure.

Finally, the scenario-specific ECL estimates are combined using probability-weighted macroeconomic scenarios to produce the portfolio's IFRS 9 Expected Credit Loss. This forward-looking approach enables financial institutions to quantify the impact of changing economic conditions on credit risk and determine the appropriate level of impairment provisions.